In [271]:
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Annotated,Literal
from langchain_core.messages import HumanMessage,BaseMessage,SystemMessage
from langgraph.graph.message import add_messages
# from langchain_core.tools import tool
# from langgraph.prebuilt import ToolNode,tools_condition
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import MemorySaver 
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [272]:
load_dotenv()
llm=ChatGroq(model="qwen/qwen3-32b")

In [273]:
class EmailState(TypedDict):
    topic:str
    plan:str
    draft:str
    status:Literal["approved","not_approved"]
    feedback:str
    review:str
    attempts:int
    sent:str
    max_attempts:3
    # messages:Annotated[list[BaseMessage],add_messages]

In [274]:
class ReviewSchema(BaseModel):
    status:Literal["approved","not_approved"]=Field(description="Status of the email approved or not")
    feedback:str=Field(description="Feedback about the mail")

In [275]:
class MailSchema(BaseModel):
    mail:str =Field(description="The mail , that is required")

In [276]:
def planner(state:EmailState):
    topic=state['topic']
    res=llm.invoke([
        SystemMessage(content="You are a planner agent. Given a topic , create a brief plan : purpose,keypoints,tone,etc necessary fields"),
        HumanMessage(content=topic)
        ]).content
    
    return {"plan": res}

def drafter(state:EmailState):
    topic=state['topic']
    plan=state["plan"]
    res=llm.with_structured_output(MailSchema).invoke([
        SystemMessage(content=f"You are a email writer agent, draft clean email"),
        HumanMessage(content=f"topic - {topic} \n plan - {plan}")
        ]).mail
    
    return {"draft":res}

def reviewer(state:EmailState):
    topic=state['topic']
    plan=state["plan"]
    draft=state['draft']
    res=llm.with_structured_output(ReviewSchema).invoke([
        SystemMessage(content="You are a strict email quality reviewer agent, review the mail and provide a feedback"),
        HumanMessage(content=f"Topic - {topic} \n plan - {plan} \n mail - {draft}")
    ])
    return {"status":res.status,"feedback":res.feedback}

def condition(state:EmailState):
    if(state['status']=="approved" or state["attempts"]>state['max_attempts']):
        return "approved"
    return "not_approved"

def optimizer(state:EmailState):
    topic=state['topic']
    draft=state['draft']
    feedback=state['feedback']
    res=llm.with_structured_output(MailSchema).invoke([
        SystemMessage(content='You are a professional email optimizer agent , improve a provided mail according to the feedback provided'),
        HumanMessage(content=f"Topic - {topic} , draft - {draft} , feedback - {feedback}")
    ]).mail
    return {"draft":res,"attempts":state['attempts']+1}

def sender(state:EmailState):
    return {'sent':"Mail sent successfully"}

In [277]:
graph=StateGraph(EmailState)
checkpointer=MemorySaver()
# Nodes-----------------------
graph.add_node("planner",planner)
graph.add_node("drafter",drafter)
graph.add_node("reviewer",reviewer)
graph.add_node("optimizer",optimizer)
graph.add_node("sender",sender)

# Edges-----------------------
graph.add_edge(START,"planner")
graph.add_edge("planner","drafter")
graph.add_edge("drafter",'reviewer')
graph.add_conditional_edges("reviewer",condition,{"approved":"sender","not_approved":"optimizer"})
graph.add_edge("optimizer",'reviewer')
graph.add_edge('sender',END)


workflow=graph.compile(checkpointer=checkpointer)


In [278]:
res=workflow.invoke({'topic':"Coldmail for internship","attempts":0,"max_attempts":3},config={"configurable":{"thread_id":"1"}})

In [279]:
res

{'topic': 'Coldmail for internship',
 'plan': '<think>\nOkay, the user needs to create a cold email for an internship. Let me start by understanding what a cold email is. It\'s an unsolicited message to someone you don\'t know, so it has to be concise, professional, and make a good first impression. The main goal is to secure an internship opportunity.\n\nFirst, I should outline the purpose. The email should introduce the user, express interest in the internship, highlight their relevant skills and experiences, and request a meeting or response. It\'s important to make it clear why they\'re reaching out and what they can offer.\n\nNext, key points. The subject line is crucial because it\'s the first thing the recipient sees. It needs to be catchy and specific. Then, the greeting should be formal. The introduction should briefly state who they are and their connection to the company. Highlighting relevant skills and experiences is next, showing how they align with the internship role. T

In [280]:
res["sent"]

'Mail sent successfully'

In [281]:
res['draft']

'Subject: Eager to Support [Project Name] as [Your Name]\n\nDear [Hiring Manager’s Name],\n\nI’m [Your Full Name], a [Your Degree] student at [University] specializing in [Relevant Field]. Your work on [Company-Specific Project/Initiative] has inspired me to contribute my skills in [Skill 1], [Skill 2], and [Skill 3] to advance [Company]’s mission in [Specific Area].\n\nMy academic project [Quantified Achievement: e.g., "a Python tool reducing data processing time by 35%"] and coursework in [Relevant Course] have prepared me to [Specific Task from Job Description]. I was particularly impressed by your team’s recent [Company-Specific Article/Whitepaper] on [Topic], which aligns with my expertise in [Relevant Skill/Interest]. \n\nWould you be available for a brief 10-minute call this week to explore how my background could support [Specific Department/Project]? I’ve attached my resume and [LinkedIn/Portfolio] for your review. I’d welcome the opportunity to discuss this further.\n\nThank 

In [282]:
res

{'topic': 'Coldmail for internship',
 'plan': '<think>\nOkay, the user needs to create a cold email for an internship. Let me start by understanding what a cold email is. It\'s an unsolicited message to someone you don\'t know, so it has to be concise, professional, and make a good first impression. The main goal is to secure an internship opportunity.\n\nFirst, I should outline the purpose. The email should introduce the user, express interest in the internship, highlight their relevant skills and experiences, and request a meeting or response. It\'s important to make it clear why they\'re reaching out and what they can offer.\n\nNext, key points. The subject line is crucial because it\'s the first thing the recipient sees. It needs to be catchy and specific. Then, the greeting should be formal. The introduction should briefly state who they are and their connection to the company. Highlighting relevant skills and experiences is next, showing how they align with the internship role. T